# W7 Homework — Your Topic Through the Workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week07/W7_hw_own_topic.ipynb)

**Goal.** Run the research workflow on a topic from your own research interests,
extend its checklist with two checks of your own design, and submit the scored
report. Nothing new to build — the first half exercised in your own domain.

The path: setup → the condensed workflow (plan → research → draft → edit) → your
topic ✍️ → your two checks ✍️ → the scored run → completion.

*Runtime:* ~40 minutes plus reading your own report. Due before the W8 midterm.
Reference answers: `labs/checkpoints/week07/solution.py`, published after the homework deadline.


## 1. Setup

*Do:* run the three cells; the last must print `ready`.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


In [ ]:
import aisuite

client = aisuite.Client()


def ask(prompt, system=None, temperature=0.0, **kwargs):
    """Single prompt -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=MODEL, messages=messages,
                                              temperature=temperature, **kwargs)
    return response.choices[0].message.content


print(ask("Reply with exactly: ready"))


## 2. The Workflow, Condensed

The lab's four stages in one pass: a planner decomposes the topic, a research call
gathers sources (keyless arXiv, with an offline fallback so the notebook runs
anywhere), a writer drafts, and the editor critiques and revises. All prompts are
the published reference versions — the homework's fill-ins are the topic and the
checks, not the machinery.

*Do:* run the cell; nothing prints yet — the run happens in Section 4.


In [ ]:
import json
import re

import requests
import xml.etree.ElementTree as ET

MOCK_SOURCES = [
    {"title": "A Survey of LLM Agents", "url": "https://arxiv.org/abs/2401.00001"},
    {"title": "Measuring Agent Reliability", "url": "https://arxiv.org/abs/2402.00002"},
    {"title": "Tool-Use Benchmarks", "url": "https://arxiv.org/abs/2403.00003"},
]


def arxiv_search(query: str, max_results: int = 4) -> list:
    """Keyless arXiv search; falls back to a mock corpus when offline."""
    try:
        r = requests.get("http://export.arxiv.org/api/query",
                         params={"search_query": f"all:{query}",
                                 "max_results": max_results}, timeout=15)
        atom = "{http://www.w3.org/2005/Atom}"
        out = [{"title": e.find(atom + "title").text.strip(),
                "url": e.find(atom + "id").text.strip()}
               for e in ET.fromstring(r.text).findall(atom + "entry")]
        if not out:
            print("NOTE: arXiv returned nothing for this query — using mock "
                  "fallback sources; broaden or rephrase your topic.")
        return out or MOCK_SOURCES
    except Exception:
        print("NOTE: arXiv unreachable — using mock fallback sources.")
        return MOCK_SOURCES


PLANNER_PROMPT = (
    "Decompose the research topic into 3 ordered steps: one research step, one "
    "drafting step, one editing step. Reply with a JSON list of step strings only.\n"
    "Topic: {topic}")

WRITER_PROMPT = (
    "Write a research report in Markdown on the topic below, using ONLY the "
    "sources given. Requirements: '## ' section headings; every claim cites a "
    "source with its full URL; a closing section naming open questions.\n\n"
    "Topic: {topic}\n\nSources:\n{sources}")

EDITOR_CRITIQUE_PROMPT = (
    "You are an editor for research reports. Return numbered, actionable feedback "
    "on the report below, judged against structure (clear Markdown sections), "
    "grounding (claims tied to cited URLs), and length (concise, no padding). "
    "Do not rewrite.\n\nReport:\n{report}")

EDITOR_REVISE_PROMPT = (
    "Revise the report to address every numbered point in the feedback. Keep all "
    "citations with full URLs. Return the complete revised report only.\n\n"
    "Report:\n{report}\n\nFeedback:\n{feedback}")


def run_workflow(topic, verbose=True):
    """Topic -> (plan, revised report). Four model calls, one search.

    The plan is recorded for your report, not dispatched — the routing executor
    lives in the lab; this condensed pipeline is fixed.
    """
    raw_plan = ask(PLANNER_PROMPT.format(topic=topic))
    steps = json.loads(re.sub(r"^```[a-z]*\s*|\s*```$", "", raw_plan.strip()))
    sources = arxiv_search(topic)
    source_text = "\n".join(f"- {s['title']} — {s['url']}" for s in sources)
    draft = ask(WRITER_PROMPT.format(topic=topic, sources=source_text))
    feedback = ask(EDITOR_CRITIQUE_PROMPT.format(report=draft))
    revised = ask(EDITOR_REVISE_PROMPT.format(report=draft, feedback=feedback))
    if verbose:
        print("PLAN:", steps, "\n")
        print("FEEDBACK:", feedback[:300], "...\n")
    return steps, revised


## 3. Your Topic and Your Checks ✍️

Two fill-ins.

**The topic.** Replace the starter with a topic from your own research area —
specific enough that the sources differ from your neighbor's ("reward hacking in
RLHF for code models", not "AI").

**Two checks of your own.** The given checklist verifies structure. Add two named
binary checks that verify what *you* care about in a report on your topic.
Requirements: each is `("name", lambda report: <bool>)`; each must be checkable
from the report text alone (notes Ch. 5: a critique question answerable from the
output is what qualifies).

Hints — shapes that work: a term that must appear ("mentions at least one named
benchmark": `lambda r: bool(re.search(r"bench", r, re.I))`); a count ("cites at
least 3 distinct URLs"); a structural fact ("has an open-questions section").

A good check is one a competent report on your topic would pass — when a check
fails, decide whether the prompt or the check is at fault before changing either.
If the scored run prints the mock-fallback notice, your query found nothing on
arXiv; broaden the topic phrasing.


In [ ]:
STARTER_TOPIC = "test-time compute for LLM agents"

### FILL IN (START) ###
TOPIC = STARTER_TOPIC          # starter — replace with your own research topic

MY_CHECKS = [
    # ("name of the check", lambda report: bool(...)),
]
### FILL IN (END) ###

BASE_CHECKS = [
    ("report is substantial (>= 800 chars)", lambda r: len(r) >= 800),
    ("has Markdown section headings", lambda r: "## " in r),
    ("cites >= 2 distinct URLs",
     lambda r: len(set(re.findall(r"https?://\S+", r))) >= 2),
]
CHECKS = BASE_CHECKS + list(MY_CHECKS)
print(f"{len(CHECKS)} checks registered")


## 4. The Scored Run

One full pass on your topic, then every check, then the judge (a variant of the lab's rubric:
grounding, structure, concision — 1 to 5). Target: **all checks PASS and judge ≥ 4**.

*Do:* run the cell, then read your own report against the feedback — would you
sign it?


In [ ]:
plan_steps, report = run_workflow(TOPIC)

check_results = {name: bool(fn(report)) for name, fn in CHECKS}
for name, ok in check_results.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {name}")

JUDGE_PROMPT = (
    "Grade the research report on three criteria - grounding (claims tied to "
    "cited URLs), structure (clear sections), concision. One sentence of "
    "justification, then exactly 'SCORE: <n>' with n from 1 to 5.\n\nReport:\n{report}")
judge_reply = ask(JUDGE_PROMPT.format(report=report))
match = re.search(r"SCORE:\s*(\d)", judge_reply)
judge_score = int(match.group(1)) if match else 0
print(f"\njudge: {judge_score}/5")


## 5. Completion Check


In [ ]:
completion = {
    "own topic set (starter replaced)": TOPIC != STARTER_TOPIC,
    "two own checks added": len(MY_CHECKS) >= 2,
    "all checks PASS": all(check_results.values()),
    "judge score >= 4": judge_score >= 4,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nHOMEWORK COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")
